# 表面缺陷模型 · 合并重训 notebook（自包含）

运行环境：**ModelScope / 魔搭** 云端 notebook（有 GPU）。
本 notebook **自包含**完成：
1. 修正云端 `raw_mine_tiles` 的错误类映射，生成 `raw_mine_tiles_fixed`；
2. 把修正切片 + 原始 40 张实拍整图合并进 `dataset/train`；
3. 重建 7 类 `data.yaml` + 生成 val 分裂；
4. 从 `last_surface_best.pt` **暖启动**合并训练（源域 + 实拍域）；
5. 导出 ONNX(裸头) + CoreML 部署包。

⚠️ 关键约定：
- 暖启动点必须是 **`last_surface_best.pt`**（surface 7 类模型自身权重），
  **不是** `last120.pt`（那是主模型 5 类，类目不对）。
- `optimizer` 必须显式 `"SGD"`，否则 auto 模式会无视 `lr0` 用 0.01 把暖启动搅乱。
- CoreML 导出后 **保持默认 BGR(colorSpace=20)**，切勿手改模型文件里的 colorSpace 为 RGB。

训练完把 `runs/surface_weld_yolov8n/weights/best.onnx` 与 `WeldDefectSurfaceModel.mlpackage`
下载回本机，重跑 `baseline_rawmine40_tiled.py` 闸门（目标 召回≥70% / 误报<10）再进阶段三。

In [ ]:
import os, shutil, random, glob

# ModelScope 云端工作区根；本地镜像用绝对路径兜底
DATA_ROOT = "/mnt/workspace/surfacedetecttrain"
if not os.path.isdir(DATA_ROOT):
    DATA_ROOT = os.path.abspath("surfacedetecttrain")
print("DATA_ROOT =", DATA_ROOT)

CLASSES = ["porosity","crack","overlap","spatters","good_weld","undercut","unfused"]
RAW_MINE = ["porosity","crack","overlap","spatters","good_weld"]  # raw_mine 真实 GT 仅这 5 类


## Step 1 · 修正云端切片标签（坏映射 -> 正确类）

In [ ]:
# 云端 raw_mine_tiles 的标签是「坏映射」(overlap->undercut 等)，须修正后才能并回训练。
# 坏 tile class -> 正确 class 的逆映射（已用 mine_001/012/004 三张图逐一核对）
REMAP = {0:0, 1:1, 2:3, 5:2, 6:4}

src_tiles = os.path.join(DATA_ROOT, "raw_mine_tiles")
fixed_dir = os.path.join(DATA_ROOT, "raw_mine_tiles_fixed")
fimgs = os.path.join(fixed_dir, "images"); flabs = os.path.join(fixed_dir, "labels")
os.makedirs(fimgs, exist_ok=True); os.makedirs(flabs, exist_ok=True)

n_fixed = 0
for f in os.listdir(os.path.join(src_tiles, "images")):
    if not f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
        continue
    shutil.copy2(os.path.join(src_tiles, "images", f), os.path.join(fimgs, f))
    lab = os.path.splitext(f)[0] + ".txt"
    src_lab = os.path.join(src_tiles, "labels", lab)
    if os.path.exists(src_lab):
        with open(src_lab) as fin, open(os.path.join(flabs, lab), "w") as fout:
            for line in fin:
                p = line.split()
                if not p:
                    continue
                c = int(float(p[0]))
                c2 = REMAP.get(c, c)
                fout.write(" ".join([str(c2)] + p[1:]) + "\n")
        n_fixed += 1
print(f"fixed tiles regenerated: {n_fixed} -> {fixed_dir}")


## Step 2 · 合并进 dataset/train（切片 + 原始整图）

In [ ]:
train_img = os.path.join(DATA_ROOT, "dataset", "train", "images")
train_lab = os.path.join(DATA_ROOT, "dataset", "train", "labels")
os.makedirs(train_img, exist_ok=True); os.makedirs(train_lab, exist_ok=True)

def merge(src_img, src_lab):
    n = 0
    if not os.path.isdir(src_img):
        print("  SKIP (missing):", src_img); return 0
    for f in os.listdir(src_img):
        if not f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
            continue
        shutil.copy2(os.path.join(src_img, f), os.path.join(train_img, f))
        lab = os.path.splitext(f)[0] + ".txt"
        s = os.path.join(src_lab, lab)
        if os.path.exists(s):
            shutil.copy2(s, os.path.join(train_lab, os.path.splitext(f)[0] + ".txt"))
        n += 1
    return n

# 1) 151 张已修正标签的切片 -> 训练 tile 推理路径
n_tiles = merge(os.path.join(DATA_ROOT, "raw_mine_tiles_fixed", "images"),
               os.path.join(DATA_ROOT, "raw_mine_tiles_fixed", "labels"))
# 2) 40 张原始实拍整图 -> 训练 whole-image 路径（标签已是正确 0~4 类）
n_full = merge(os.path.join(DATA_ROOT, "raw_mine", "images"),
              os.path.join(DATA_ROOT, "raw_mine", "labels"))
print(f"merged: tiles={n_tiles}, full={n_full}")
print("train images total:", len(os.listdir(train_img)))


## Step 3 · 重建 data.yaml + val 分裂

In [ ]:
import yaml
random.seed(42)
train_img = os.path.join(DATA_ROOT, "dataset", "train", "images")
train_lab = os.path.join(DATA_ROOT, "dataset", "train", "labels")
all_imgs = [f for f in os.listdir(train_img) if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]

# 生成 val 分裂（从合并后 train 随机抽 10% 作 val，用于训练中 mAP 监控）
val_ratio = 0.10
val_imgs = set(random.sample(all_imgs, max(1, int(len(all_imgs) * val_ratio))))
val_img_dir = os.path.join(DATA_ROOT, "dataset", "val", "images")
val_lab_dir = os.path.join(DATA_ROOT, "dataset", "val", "labels")
os.makedirs(val_img_dir, exist_ok=True); os.makedirs(val_lab_dir, exist_ok=True)
for f in val_imgs:
    shutil.copy2(os.path.join(train_img, f), os.path.join(val_img_dir, f))
    lab = os.path.splitext(f)[0] + ".txt"
    s = os.path.join(train_lab, lab)
    if os.path.exists(s):
        shutil.copy2(s, os.path.join(val_lab_dir, lab))
print("val images:", len(val_imgs))

data = {
    "path": "dataset",
    "train": "train/images",
    "val": "val/images",
    "nc": len(CLASSES),
    "names": CLASSES,
}
yaml_path = os.path.join(DATA_ROOT, "dataset", "data.yaml")
with open(yaml_path, "w") as fp:
    yaml.safe_dump(data, fp, allow_unicode=True)
print("wrote", yaml_path)


In [ ]:
from collections import Counter
cnt = Counter()
for f in os.listdir(train_lab):
    for line in open(os.path.join(train_lab, f)):
        p = line.split()
        if p:
            try: cnt[int(p[0])] += 1
            except: pass
print("train label class distribution (index: name = count):")
for i, c in enumerate(CLASSES):
    print(f"  {i} {c}: {cnt.get(i, 0)}")
assert all(cnt.get(i, 0) > 0 for i in range(7)), "存在某类样本为 0，训练会缺信号"
print("OK: 仅 0~4 类，无 undercut/unfused 污染")


## Step 4 · 合并训练（暖启动 last_surface_best.pt）

In [ ]:
from ultralytics import YOLO
import subprocess, sys
# 页面刷新后 kernel 可能重置，确保依赖在
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics", "coremltools==9.0", "numpy==2.3.5"], check=False)

MODEL_IN  = os.path.join(DATA_ROOT, "last_surface_best.pt")   # 暖启动：surface 自身最佳权重（非 last120.pt）
RUN_DIR   = os.path.join(DATA_ROOT, "runs", "surface_weld_yolov8n")
if os.path.exists(RUN_DIR):
    shutil.rmtree(RUN_DIR)   # 清掉旧 run，避免 glob 取到废权重

m = YOLO(MODEL_IN)
m.train(
    data=os.path.join(DATA_ROOT, "dataset", "data.yaml"),
    imgsz=640, batch=32, epochs=120,
    optimizer="SGD",          # 必须显式，否则 auto 无视 lr0
    lr0=0.001, lrf=0.01,      # 低 lr 暖启动微调，best 通常在中后期
    warmup_epochs=3, momentum=0.9, weight_decay=0.0005,
    patience=50,
    name="surface_weld_yolov8n",
    copy_paste=0.4, mixup=0.1, fliplr=0.5, flipud=0.1,
    degrees=5.0, scale=0.5, shear=1.0, perspective=0.0005,
    hsv_s=0.9, hsv_v=0.5,
    project=os.path.join(DATA_ROOT, "runs"),
    exist_ok=True,
)


## Step 5 · 导出 ONNX(裸头) + CoreML

In [ ]:
from ultralytics import YOLO
best = os.path.join(DATA_ROOT, "runs", "surface_weld_yolov8n", "weights", "best.pt")
m = YOLO(best)
# 裸头 (1,11,8400)，nms=False 以便本机 onnxruntime 复刻闸门
m.export(format="onnx", imgsz=640, dynamic=False, nms=False, opset=17, simplify=True)
print("ONNX ->", os.path.join(DATA_ROOT, "runs", "surface_weld_yolov8n", "weights", "best.onnx"))


In [ ]:
from ultralytics import YOLO
import shutil as _sh
best = os.path.join(DATA_ROOT, "runs", "surface_weld_yolov8n", "weights", "best.pt")
m = YOLO(best)
# nms=False；保持默认 BGR(20)，勿手改 colorSpace
m.export(format="coreml", imgsz=640, nms=False)

src = os.path.join(DATA_ROOT, "runs", "surface_weld_yolov8n", "weights", "best.mlpackage")
dst = os.path.join(DATA_ROOT, "WeldDefectSurfaceModel.mlpackage")
if os.path.exists(src):
    if os.path.exists(dst): _sh.rmtree(dst)
    _sh.move(src, dst)
    print("CoreML ->", dst)


## Step 6 · 回本机验证 + 判定

In [ ]:
# ---- 训练+导出完成后，回本机做闸门验证 ----
# 1) 从 ModelScope 把这两个文件下载到本机 D:\workbuddy\workbuddy学习\surfacedetecttrain\ ：
#      /mnt/workspace/surfacedetecttrain/runs/surface_weld_yolov8n/weights/best.onnx
#      /mnt/workspace/surfacedetecttrain/WeldDefectSurfaceModel.mlpackage
#    （覆盖旧版 best.onnx / 旧 mlpackage）
#
# 2) 本机执行（Git Bash）：
#      cd /d/workbuddy/workbuddy学习
#      C:/Users/Administrator/.workbuddy/binaries/python/envs/default/Scripts/python.exe \
#          weld_fatigue_checker/ml/baseline_rawmine40_tiled.py --onnx surfacedetecttrain/best.onnx
#
# 3) 判定：召回≥70% 且 误报<10 → PASS，进阶段三部署 + CI 推 R3 Swift；
#         否则 FAIL，需补充更多 raw_mine 实拍标注后重训。